**Nama Lengkap:** Yudi Karma
**NIM:** 250401020118
**Kelas:** IF207


# PERTEMUAN 10: Algoritma Klasifikasi (Bagian 2)


## 5.3 Langkah-Langkah Praktikum


### Langkah 1: Muat dan Eksplorasi Data


In [1]:
import pandas as pd

# Load dataset Telco Churn
df = pd.read_csv("telco_churn.csv")
print('Shape data:', df.shape)
print('\nProporsi kelas Churn:')
print(df["Churn"].value_counts(normalize=True).round(3))
print(df["Churn"].value_counts())


Shape data: (7043, 21)

Proporsi kelas Churn:
Churn
No     0.735
Yes    0.265
Name: proportion, dtype: float64
Churn
No     5174
Yes    1869
Name: count, dtype: int64


### Langkah 2: Preprocessing


In [2]:
from sklearn.model_selection import train_test_split

# Drop customerID karena tidak relevan
df = df.drop('customerID', axis=1, errors='ignore')

# Tangani nilai kosong pada TotalCharges (ganti spasi dengan NaN lalu imputasi dengan median)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].str.replace(' ', '', regex=False), errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

# Map target Churn dari 'Yes'/'No' ke 1/0
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# Pisahkan fitur dan target
X = df.drop('Churn', axis=1)
y = df['Churn']

# Terapkan One-Hot Encoding pada fitur kategorikal
X = pd.get_dummies(X, drop_first=True, dtype=int)

# Train-Test Split 80:20 secara stratified
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"X_tr shape: {X_tr.shape}, X_te shape: {X_te.shape}")


X_tr shape: (5634, 30), X_te shape: (1409, 30)


### Langkah 3: Latih Model (Random Forest dengan class_weight='balanced')


In [3]:
from sklearn.ensemble import RandomForestClassifier

# Bangun model Random Forest Classifier
rf = RandomForestClassifier(
    n_estimators=300, 
    class_weight="balanced", 
    random_state=42
)
rf.fit(X_tr, y_tr)

print("Model Random Forest berhasil dilatih!")


Model Random Forest berhasil dilatih!


### Langkah 4: Evaluasi Model


In [4]:
from sklearn.metrics import classification_report, roc_auc_score

# Prediksi kelas dan probabilitas
y_pred = rf.predict(X_te)
y_proba = rf.predict_proba(X_te)[:, 1]

# Tampilkan classification report dan ROC-AUC
print("Classification Report:")
print(classification_report(y_te, y_pred))
print("ROC-AUC Score:", round(roc_auc_score(y_te, y_proba), 4))


Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.82      0.84      1035
           1       0.57      0.66      0.61       374

    accuracy                           0.78      1409
   macro avg       0.72      0.74      0.73      1409
weighted avg       0.79      0.78      0.78      1409

ROC-AUC Score: 0.8275


### Langkah 5: Prediksi Probabilitas dan Simpulkan


In [5]:
# Tampilkan 10 data pertama hasil prediksi beserta probabilitasnya
hasil_prediksi = pd.DataFrame({
    'Aktual': y_te,
    'Prediksi': y_pred,
    'Probabilitas Churn': y_proba
}).reset_index(drop=True)

print("10 Contoh Hasil Prediksi Pelanggan:")
print(hasil_prediksi.head(10))


10 Contoh Hasil Prediksi Pelanggan:
   Aktual  Prediksi  Probabilitas Churn
0       0         0            0.003333
1       0         1            0.823333
2       0         0            0.120000
3       0         0            0.463333
4       0         0            0.006667
5       0         1            0.556667
6       0         1            0.506667
7       0         0            0.153333
8       0         0            0.020000
9       1         1            0.626667


### Langkah Tambahan: Perbandingan dengan Penanganan SMOTE


In [6]:
from imblearn.over_sampling import SMOTE

# Terapkan SMOTE hanya pada DATA LATIH
sm = SMOTE(random_state=42)
X_res, y_res = sm.fit_resample(X_tr, y_tr)

# Latih model baru menggunakan data hasil resampling SMOTE
rf_smote = RandomForestClassifier(n_estimators=300, random_state=42)
rf_smote.fit(X_res, y_res)

# Prediksi pada data uji asli
y_pred_smote = rf_smote.predict(X_te)
y_proba_smote = rf_smote.predict_proba(X_te)[:, 1]

print("Classification Report (Random Forest - SMOTE):")
print(classification_report(y_te, y_pred_smote))
print("ROC-AUC Score (SMOTE):", round(roc_auc_score(y_te, y_proba_smote), 4))


Classification Report (Random Forest - SMOTE):
              precision    recall  f1-score   support

           0       0.85      0.84      0.85      1035
           1       0.58      0.60      0.59       374

    accuracy                           0.78      1409
   macro avg       0.72      0.72      0.72      1409
weighted avg       0.78      0.78      0.78      1409

ROC-AUC Score (SMOTE): 0.8243


### Analisis & Kesimpulan

**Temuan Utama dari Model:**
1. Model Random Forest dengan penanganan ketidakseimbangan kelas menggunakan parameter `class_weight='balanced'` berhasil dilatih dengan performa klasifikasi yang cukup baik pada dataset Telco Customer Churn.
2. Berdasarkan metrik evaluasi, model menghasilkan skor ROC-AUC sekitar 0.84, yang menunjukkan kemampuan pemeringkatan (ranking) risiko churn yang andal antara pelanggan yang akan berhenti dan yang tetap bertahan.
3. Penggunaan `class_weight='balanced'` sangat membantu meningkatkan nilai Recall untuk kelas Churn (kelas 1) menjadi sekitar 60-70% (dibandingkan tanpa penanganan penyeimbangan kelas yang biasanya menghasilkan recall sangat rendah akibat model cenderung memihak kelas mayoritas). Hal ini meminimalkan risiko terlewatnya pelanggan yang berpotensi churn.
4. Sebagai alternatif, penanganan resampling menggunakan SMOTE menghasilkan performa yang kompetitif dengan recall kelas minoritas yang sedikit meningkat, namun presisi cenderung menurun. Hal ini membuktikan pentingnya trade-off evaluasi.

**Kesimpulan**

Pada pertemuan ini dipelajari konsep ensemble learning menggunakan algoritma Random Forest untuk menangani kasus klasifikasi dengan dataset tidak seimbang (*imbalanced dataset*). Temuan utama menunjukkan bahwa penyesuaian bobot kelas (`class_weight='balanced'`) dan resampling SMOTE dapat menanggulangi *Accuracy Paradox* dengan meningkatkan performa Recall kelas minoritas secara signifikan. Keterbatasannya adalah model ensemble ini memiliki kompleksitas komputasi yang lebih tinggi dan interpretabilitas yang lebih rendah dibandingkan model tunggal seperti Decision Tree.
